# MinIO - Uitgebreide Uitleg en Voorbeelden

In dit notebook behandelen we het gebruik van **MinIO**, een object storage service vergelijkbaar met AWS S3. We bekijken hoe we MinIO kunnen gebruiken via:

- CLI (Command Line Interface)
- Python (via `minio` library)
- Web UI

We behandelen de volgende onderwerpen:

1. Connecteren met MinIO (CLI en Python)
2. Bestanden schrijven naar MinIO (CLI en Python)
3. Bestanden lezen van MinIO (Web UI, CLI, Python)
4. Bestanden aanpassen in MinIO (rechten en instellingen)
5. MinIO instellingen aanpassen (CLI en Python)


## 1. Connecteren met MinIO


### 1.1 MinIO CLI

Op de cluster is de MinIO cluster geinstalleerd waarvan je hier een guide kan vinden: https://docs.min.io/docs/minio-client-quickstart-guide.html

In de shell waarin je cli-commando's wil gebruiken moet je eerst dit doen (na het sluiten van de shell instantie is het alias gereset):

In [1]:
!mc alias set minio_cli http://minio1:9000 bigdata bigdata123

]11;?\mc: Configuration written to `/root/.mc/config.json`. Please update your access credentials.
mc: Successfully created `/root/.mc/share`.
mc: Initialized share uploads `/root/.mc/share/uploads.json` file.
mc: Initialized share downloads `/root/.mc/share/downloads.json` file.
Added `minio_cli` successfully.


- `minio_cli` is de alias voor de server
- `http://minio1:9000` is het endpoint (in dit geval 1 van de nodes van de cluster)
- `bigdata`/`bigdata123` zijn de access key en secret key

Of het correct gebeurd is kan je als volgt controleren

In [2]:
!mc alias list

]11;?\gcs      
  URL       : https://storage.googleapis.com
  AccessKey : YOUR-ACCESS-KEY-HERE
  SecretKey : YOUR-SECRET-KEY-HERE
  API       : S3v2
  Path      : dns
  Src       : /root/.mc/config.json

local    
  URL       : http://localhost:9000
  AccessKey : 
  SecretKey : 
  API       : 
  Path      : auto
  Src       : /root/.mc/config.json

minio_cli
  URL       : http://minio1:9000
  AccessKey : bigdata
  SecretKey : bigdata123
  API       : s3v4
  Path      : auto
  Src       : /root/.mc/config.json

play     
  URL       : https://play.min.io
  AccessKey : Q3AM3UQ867SPQQA43P2F
  SecretKey : zuf+tfteSlswRu7BJ86wekitnifILbZam1KYY3TG
  API       : S3v4
  Path      : auto
  Src       : /root/.mc/config.json

s3       
  URL       : https://s3.amazonaws.com
  AccessKey : YOUR-ACCESS-KEY-HERE
  SecretKey : YOUR-SECRET-KEY-HERE
  API       : S3v4
  Path      : dns
  Src       : /root/.mc/config.json



### 1.2 Python (MinIO SDK)

Verbind met de server door gebruik te maken van het minio package:

In [12]:
from minio import Minio

client = Minio(
    endpoint='minio1:9000', # minio1 enkel gekend op de containers
    access_key='bigdata',
    secret_key='bigdata123',
    secure=False,
    region='eu-west-1'
)

client.list_buckets()

[Bucket(name='01-mapreduce', creation_date=datetime.datetime(2026, 3, 3, 13, 11, 43, 563000, tzinfo=datetime.timezone.utc)),
 Bucket(name='mijnclibucket', creation_date=datetime.datetime(2026, 3, 6, 13, 6, 30, 467000, tzinfo=datetime.timezone.utc)),
 Bucket(name='mijnpythonbucket', creation_date=datetime.datetime(2026, 2, 24, 14, 8, 57, 985000, tzinfo=datetime.timezone.utc))]

## 2. Bestanden schrijven naar MinIO


### 2.1 CLI

Maak een bucket aan:

In [5]:
!mc rb minio_cli/mijnclibucket --force

]11;?\Removed `minio_cli/mijnclibucket` successfully.


In [6]:
!mc mb minio_cli/mijnclibucket

]11;?\Bucket created successfully `minio_cli/mijnclibucket`.


Upload een bestand:

In [7]:
!mc cp ulysses.txt minio_cli/mijnclibucket/

...lysses.txt: 1.51 MiB / 1.51 MiB ┃▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓┃ 24.18 MiB/s 0s

Surf nu naar de WebUI van MinIO (http://localhost:9001), login met bovenstaande credentials en bekijk de bucket. Zie je het bestand staan?

### 2.2 Python

Dezelfde stappen kunnen we ook doen in een python script als volgt:

In [15]:
bucket_name = 'mijnpythonbucket'
if not client.bucket_exists(bucket_name):
    client.make_bucket(bucket_name)

client.fput_object(bucket_name, 'map/notebooks.txt', 'davinci_notebooks.txt')

ObjectWriteResult(bucket_name='mijnpythonbucket', object_name='map/notebooks.txt', version_id=None, etag='11fbe53f92ce6e30dc8117ce46cdd550', http_headers=HTTPHeaderDict({'Accept-Ranges': 'bytes', 'Content-Length': '0', 'ETag': '"11fbe53f92ce6e30dc8117ce46cdd550"', 'Server': 'MinIO', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'Vary': 'Origin, Accept-Encoding', 'X-Amz-Id-2': 'e0c385c033c4356721cc9121d3109c9b9bfdefb22fd2747078acd22328799e36', 'X-Amz-Request-Id': '189A459AA7EFAF18', 'X-Content-Type-Options': 'nosniff', 'X-Ratelimit-Limit': '1015', 'X-Ratelimit-Remaining': '1015', 'X-Xss-Protection': '1; mode=block', 'Date': 'Fri, 06 Mar 2026 14:05:33 GMT'}), last_modified=None, location=None)

Ga opnieuw naar de webUI en bekijk of het correct uitgevoerd is

## 3. Bestanden lezen van MinIO


### 3.1 Via Web UI

- Open je browser en ga naar `http://localhost:9000`
- Login met access key / secret key
- Je kunt bestanden downloaden of bekijken door op het bucket te klikken.


### 3.2 Via CLI

Oplijsten van bestanden kan via het ls-commando:

In [8]:
# Lijst bestanden
!mc ls minio_cli/mijnclibucket

]11;?\[2026-03-06 13:07:41 UTC] 1.5MiB STANDARD ulysses.txt


Downloaden van bestanden kan als volgt

In [9]:
#Download bestand
!mc cp minio_cli/mijnclibucket/ulysses.txt downloaded.txt

...lysses.txt: 1.51 MiB / 1.51 MiB ┃▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓┃ 52.74 MiB/s 0s

### 3.3 Via Python

In [17]:
# Download bestand
client.fget_object(bucket_name, "map/notebooks.txt", "downloaded2.txt")
# let op de volgorde van de variabelen
# zowel bij fput als bij fget:
# bucket_name, naam_op_minio, naam_lokaal

Object(bucket_name='mijnpythonbucket', object_name='map/notebooks.txt', last_modified=datetime.datetime(2026, 3, 6, 14, 5, 33, tzinfo=datetime.timezone.utc), etag='11fbe53f92ce6e30dc8117ce46cdd550', size=1428843, metadata=HTTPHeaderDict({'Accept-Ranges': 'bytes', 'Content-Length': '1428843', 'Content-Type': 'application/octet-stream', 'ETag': '"11fbe53f92ce6e30dc8117ce46cdd550"', 'Last-Modified': 'Fri, 06 Mar 2026 14:05:33 GMT', 'Server': 'MinIO', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'Vary': 'Origin, Accept-Encoding', 'X-Amz-Id-2': 'e0c385c033c4356721cc9121d3109c9b9bfdefb22fd2747078acd22328799e36', 'X-Amz-Request-Id': '189A4637AD11D2C9', 'X-Content-Type-Options': 'nosniff', 'X-Ratelimit-Limit': '1015', 'X-Ratelimit-Remaining': '1015', 'X-Xss-Protection': '1; mode=block', 'Date': 'Fri, 06 Mar 2026 14:16:48 GMT'}), version_id=None, is_latest=None, storage_class=None, owner_id=None, owner_name=None, content_type='application/octet-stream', is_delete_marker=F

In [20]:
for obj in client.list_objects(bucket_name):
    print(obj.object_name)

davinci_notebooks.txt
notebooks.txt
map/


## 4. Bestanden aanpassen in MinIO

### 🔒 Private (default)

* Enkel geauthenticeerde gebruikers kunnen het bestand zien
  * Andere krijgen een error bij het gebruiken van de url van het bestand

### 🌍 Public

* Iedereen kan het zien en downloaden.
  * Bvb via de link:
```
http://<minio-server>:9000/mybucket/image.jpg
```
* Use case: hosten statische bestanden, datasets for studenten, publieke assets voor web pagina's.

Het is mogelijk om deze rechten op bestandsniveau toe te passen maar de standaard is om het op bucketniveau te doen.

### 4.1 Rechten aanpassen (CLI)

In [10]:
# Maak bucket public
!mc policy set public minio_cli/mijnclibucket

# Maak bucket private
!mc policy set private minio_cli/mijnclibucket


]11;?\mc: Please use 'mc anonymous'
]11;?\mc: Please use 'mc anonymous'


### 4.2 Rechten aanpassen (Python)

* Principal: * → De policy gaat over iedereen, dus ook niet geauthenticeerde gebruikers.
* Action: Welke acties die vastgelegd worden in deze policy
* Resource: Over welke bucket/files het gaat

In [21]:
# Stel een policy in (voorbeeld: readonly)
policy_readonly = f'''{{
  "Version":"2012-10-17",
  "Statement":[
    {{
      "Effect":"Allow",
      "Principal":{{"AWS":["*"]}},
      "Action":[
        "s3:GetBucketLocation",
        "s3:ListBucket",
        "s3:GetObject"
      ],
      "Resource":[
        "arn:aws:s3:::{bucket_name}",
        "arn:aws:s3:::{bucket_name}/*"
      ]
    }}
  ]
}}'''

client.set_bucket_policy(bucket_name, policy_readonly)

## 5. MinIO instellingen aanpassen

MinIO heeft 2 grote categorieën instellingen:

1. **Server configuratie** (zoals credentials, storage class, logging, region, WORM, etc.)
2. **Gebruikers en groepen** (accounts, rechten, policies).

## 🔧 1. Wat kan je aanpassen?

Via `mc admin config` en `mc admin service` kan je o.a.:

* **Gebruikersbeheer**

  * Toevoegen, aanpassen, verwijderen van gebruikers.
  * Wachtwoorden wijzigen.
  * Gebruikers activeren/deactiveren.

* **Region**

  * `region.name` → instellen van de regio van je MinIO cluster.

* **Credentials**

  * `root user` en `root password` aanpassen.

* **Storage class**

  * `storage_class.standard` en `storage_class.rrs` (Reduced Redundancy Storage) configureren.

* **WORM (Write Once Read Many)**

  * `worm=on` om data immutable te maken.

* **Logging**

  * HTTP en audit logging in- of uitschakelen.

* **KMS (Key Management System)**

  * Encryptie instellen.

* **Healing / Erasure coding**

  * Parameters aanpassen voor dataherstel in distributed setups.


### 5.1 Via CLI

**Oefening:** Ga op zoek naar wat de volgende commando's doen

In [ ]:
!mc admin user add minio_cli newuser newpassword
!mc admin user enable minio_cli newuser
!mc admin user list minio_cli

!mc admin config set minio_cli identity_root_user=newroot identity_root_password=newpass123
!mc admin config set minio_cli region name="eu-west-1"
!mc admin config set minio_cli logger_http enabled=on
!mc admin config set minio_cli worm=on
!mc admin service restart minio_cli

!mc admin info myminio

### 5.2 Via Python

De MinIO Python SDK ondersteunt beperkte admin functies. Voor volledige admin functies moet je meestal CLI of de Web UI gebruiken.

Voor beperkte admin acties via Python kun je gebruik maken van `subprocess` om CLI commands aan te roepen:

```python
import subprocess

# Voorbeeld: lijst buckets via CLI
result = subprocess.run(["mc", "ls", "minio_cli"], capture_output=True, text=True)
print(result.stdout)
```

### Extra tips:
- MinIO ondersteunt AWS S3 API, dus je kunt ook `boto3` gebruiken voor interacties.
- Voor productie gebruik altijd `secure=True` en stel TLS in.
- Policies en bucket instellingen kunnen gedetailleerd worden aangepast via JSON policies.

## Oefeningen

Los de volgende oefeningen op, voor sommige oefeningen ga je de nodige functies moeten opzoeken in de documentatie:
* Oefening 1: Metadata en informatie
  * Zoek op in de documentatie welke functie de metadata van een object kan ophalen (hint: stat_object).
  * Gebruik dit om de grootte en content-type van je bestand te tonen.
* Oefening 2: Upload drie bestanden naar een bucket.
  * Verwijder één bestand met remove_object.
  * Controleer met list_objects() of hij effectief weg is.
* Oefening 3: Maak een Python-script dat het volgende doet:
  * Bucket check
    * Maak een bucket aan met een zelfgekozen naam.
    * Als de bucket al bestaat:
    * Leeg de bucket → verwijder alle bestanden, behalve 2 specifieke bestandsnamen (bijv. keep1.txt en keep2.txt).
  * Uploads
    * Controleer of keep1.txt en keep2.txt in de bucket aanwezig zijn.
    * Als een bestand ontbreekt: upload het vanuit de lokale map.


In [ ]:
# Oefening 1
# Eerst een bestand uploaden (als dat nog niet in de bucket zit)

In [ ]:
# Oefening 2
# Upload drie bestanden